# 05 · Pseudobulk differential expression per cell type — GSE216146

Chemotherapy (cisplatin) vs control (PBS), separately for each cell type.

Method: raw counts are **summed per biosample x cell type** (pseudobulk; the sample is the unit of replication, never
the cell), then PyDESeq2 with `pseudobulk.design` (`~ batch + condition`; batch absorbs the lower-quality D20 batch).
A sample enters a cell type only with >= `min_cells_per_sample` cells; a cell type is tested only with
>= `min_samples_per_group` samples per condition. Wald test, BH FDR; apeGLM-style shrunken log2 fold changes are
stored in `lfc_shrunk` (used for ranking in `06_enrichment`), the volcano plots use the unshrunken estimate.

Flags (genes are kept, but marked and not labelled in the plots):
- `ambient`: choroid plexus genes (>= `ambient_fold` higher there than in the tested cell type) and
  `annotation.ambient_genes`. Ambient RNA (e.g. *Ttr*) is higher in the chemotherapy samples (03), so these genes
  tend to look "up in chemotherapy" in every cell type.
- `other_type`: genes >= `other_type_fold` higher in another cell type (doublets / ambient from that type; e.g.
  endothelial *Cxcl12* in oligodendrocytes). Columns `cpm_cell_type`, `top_other_type`, `top_other_ratio` show why.
- `stress`: immediate-early / dissociation genes (`pseudobulk.stress_genes`).
- `technical`: genes overlapping rRNA (*Gm42418*, *AY036118*, *Lars2*) and similar library-prep artefacts
  (`pseudobulk.technical_genes`); they came out "down in chemotherapy" in 5 cell types.

QC caveats (`01_qc`, QC kept as is): the neuron pseudobulk is built from the ~40–50 % of neurons that survive the
mt cut (fewer in cisplatin), so neuron DE is weak and possibly biased towards low-mt neurons; in microglia the ribo
filter removed more cisplatin cells with high ribosomal content, which dampens ribosome/translation changes.

**Input:** `results/03_annotated.h5ad` · **Output:** `results/05_pseudobulk.h5ad`, `results/de/<cell_type>.tsv`,
`results/tables/05_de_summary.tsv`, plots `results/plots/05_*`.

In [ ]:
import gc
import os
import sys
import warnings
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
p = cfg["pseudobulk"]
de_dir = cfg["paths"]["results_dir"] / "de"
de_dir.mkdir(exist_ok=True)
scu.setup_plotting()
# lfc_shrink emits harmless overflow RuntimeWarnings from parallel workers; workers inherit this setting
os.environ["PYTHONWARNINGS"] = "ignore::RuntimeWarning"
print("pydeseq2", version("pydeseq2"))
p

## 1. Pseudobulk counts

In [ ]:
adata = sc.read_h5ad(cfg["paths"]["results_dir"] / "03_annotated.h5ad")
drop = cfg["downstream"]["exclude_cell_types"]
adata = adata[~adata.obs["cell_type"].isin(drop)].copy()
print("excluded", drop, "->", adata.n_obs, "cells")

pb = scu.pseudobulk(adata, p["sample_key"], p["group_key"], layer="counts", obs_cols=("condition", "batch"))
del adata
gc.collect()
pb.write_h5ad(cfg["paths"]["results_dir"] / "05_pseudobulk.h5ad", compression="gzip")
pb

In [ ]:
n_cells = pb.obs.pivot(index=p["group_key"], columns=p["sample_key"], values="n_cells").fillna(0).astype(int)
cond_of = pb.obs.drop_duplicates(p["sample_key"]).set_index(p["sample_key"])["condition"]
ok = n_cells >= p["min_cells_per_sample"]
factor, test_level, ref_level = p["contrast"]
n_ok = pd.DataFrame({lvl: ok.loc[:, cond_of[ok.columns] == lvl].sum(axis=1) for lvl in (ref_level, test_level)})
n_ok["tested"] = (n_ok >= p["min_samples_per_group"]).all(axis=1)
cell_types = list(n_ok.index[n_ok["tested"]])
print("tested:", cell_types)
print("skipped:", list(n_ok.index[~n_ok["tested"]]))
n_cells.join(n_ok)

In [ ]:
# Pooled CPM per cell type: reference for the ambient / other-cell-type flags
pooled = pd.DataFrame(pb.X.toarray(), index=pb.obs_names, columns=pb.var_names).groupby(pb.obs[p["group_key"]].values).sum()
cpm = pooled.div(pooled.sum(axis=1), axis=0) * 1e6
src = p["ambient_source"]
always_ambient = set(cfg["annotation"]["ambient_genes"])
gene_lists = {"stress": set(p["stress_genes"]), "technical": set(p["technical_genes"])}

def annotate_genes(genes: pd.Index, cell_type: str) -> pd.DataFrame:
    """Expression context and flags per gene: CPM in this cell type, the other cell type with the highest CPM
    and the ratio, plus comma-separated flags (ambient, other_type, stress, technical; '' = clean)."""
    here = cpm.loc[cell_type, genes]
    others = cpm.drop(index=cell_type)[genes]
    ratio_other = (others.max(axis=0) + 1) / (here + 1)
    ratio_src = (cpm.loc[src, genes] + 1) / (here + 1)
    tags = {"ambient": genes.isin(always_ambient) | ((cell_type != src) & (ratio_src >= p["ambient_fold"]).to_numpy()),
            "other_type": (ratio_other >= p["other_type_fold"]).to_numpy()}
    tags.update({name: genes.isin(lst) for name, lst in gene_lists.items()})
    tagged = pd.DataFrame(tags, index=genes)
    return pd.DataFrame({"cpm_cell_type": here.round(1), "top_other_type": others.idxmax(axis=0),
                         "top_other_ratio": ratio_other.round(1),
                         "flag": tagged.apply(lambda r: ",".join(r.index[r]), axis=1)}, index=genes)

## 2. PyDESeq2 per cell type

In [ ]:
inference = DefaultInference(n_cpus=p["n_cpus"])
results, summary = {}, []
for ct in cell_types:
    rows = (pb.obs[p["group_key"]] == ct) & (pb.obs["n_cells"] >= p["min_cells_per_sample"])
    sub = pb[rows]
    counts = pd.DataFrame(sub.X.toarray().round().astype(int), index=sub.obs[p["sample_key"]], columns=sub.var_names)
    keep = (counts >= p["min_gene_counts"]).sum(axis=0) >= p["min_gene_samples"]
    counts = counts.loc[:, keep]
    meta = sub.obs.set_index(p["sample_key"])[["condition", "batch"]].copy()
    meta["condition"] = pd.Categorical(meta["condition"], categories=[ref_level, test_level])
    design = p["design"] if meta["batch"].nunique() > 1 else "~ condition"

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dds = DeseqDataSet(counts=counts, metadata=meta, design=design, refit_cooks=True,
                           inference=inference, quiet=True)
        dds.deseq2()
        ds = DeseqStats(dds, contrast=[factor, test_level, ref_level], alpha=p["padj"], inference=inference, quiet=True)
        ds.summary()
        res = ds.results_df.copy()
        coeff = [c for c in dds.obsm["design_matrix"].columns if c.startswith(f"{factor}[T.")][0]
        ds.lfc_shrink(coeff=coeff)
    res["lfc_shrunk"] = ds.results_df["log2FoldChange"]
    res = res.join(annotate_genes(res.index, ct))
    res = res.sort_values("pvalue")
    res.to_csv(de_dir / f"{scu.slug(ct)}.tsv", sep="\t", index_label="gene")
    results[ct] = res

    sig = (res["padj"] < p["padj"]) & (res["log2FoldChange"].abs() >= p["lfc"])
    clean = sig & (res["flag"] == "")
    summary.append({"cell_type": ct, "design": design,
                    f"n_{ref_level}": int((meta["condition"] == ref_level).sum()),
                    f"n_{test_level}": int((meta["condition"] == test_level).sum()),
                    "cells": int(sub.obs["n_cells"].sum()), "genes_tested": counts.shape[1],
                    "up": int((clean & (res["log2FoldChange"] > 0)).sum()),
                    "down": int((clean & (res["log2FoldChange"] < 0)).sum()),
                    "flagged_sig": int((sig & (res["flag"] != "")).sum())})
    print(f"{ct:18s} {summary[-1]['up']:4d} up {summary[-1]['down']:4d} down  ({summary[-1]['flagged_sig']} flagged)")
    del dds, ds
    gc.collect()

summary = pd.DataFrame(summary).set_index("cell_type")
summary.to_csv(cfg["paths"]["tables_dir"] / "05_de_summary.tsv", sep="\t")
summary

## 3. Volcano plots

In [ ]:
for ct, res in results.items():
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    scu.plot_volcano(res, ax, lfc=p["lfc"], padj=p["padj"], n_label=p["n_label"], title=ct)
    scu.save_fig(fig, cfg, f"05_volcano_{scu.slug(ct)}")
    plt.close(fig)

ncol = 4
nrow = int(np.ceil(len(results) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.3 * nrow))
for ax, (ct, res) in zip(axes.flat, results.items()):
    scu.plot_volcano(res, ax, lfc=p["lfc"], padj=p["padj"], n_label=8, title=ct)
for ax in list(axes.flat)[len(results):]:
    ax.remove()
fig.tight_layout()
scu.save_fig(fig, cfg, "05_volcano_overview")
plt.show()

## 4. Overview

Significant genes per cell type (unflagged), and genes significant in several cell types.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
summary[["up", "down", "flagged_sig"]].plot(kind="bar", ax=ax, color=["#c0392b", "#2c7bb6", "0.6"], width=0.8)
ax.set_ylabel(f"genes (padj < {p['padj']}, |log2FC| >= {p['lfc']})"); ax.set_xlabel(""); ax.grid(False)
ax.tick_params(axis="x", labelrotation=45)
scu.save_fig(fig, cfg, "05_de_counts")
plt.show()

In [ ]:
sig_long = pd.concat({ct: r[(r["padj"] < p["padj"]) & (r["log2FoldChange"].abs() >= p["lfc"])]
                      [["log2FoldChange", "padj", "flag"]] for ct, r in results.items()}, names=["cell_type", "gene"]).reset_index()
shared = (sig_long.groupby("gene")
          .agg(n_cell_types=("cell_type", "size"), cell_types=("cell_type", ", ".join),
               mean_log2FC=("log2FoldChange", "mean"), flag=("flag", "first"))
          .query("n_cell_types >= 2").sort_values("n_cell_types", ascending=False))
shared.to_csv(cfg["paths"]["tables_dir"] / "05_de_shared_genes.tsv", sep="\t")
shared.head(30)

In [ ]:
del pb, results
_ = gc.collect()